# ML-04 — Search Intelligence Data Contract

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ahmedali2155/flyrank-ml-internship/blob/main/work/notebooks/w03_data_contract.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

In [13]:
HF_TOKEN = "REMOVED"

## 1. Unit of analysis + time window

*One row = one what, over which dates? State it, then verify it below.*

### 1. Unit of Analysis
One row represents one content item per client per day.

### 2. Tables Used
I use the table `fact_content_daily_performance`.

### 3. Time Window
I use data from March 2026 (month=2026-03).

### 4. Prediction Target
I aim to predict whether content will receive clicks (gsc_clicks > 0).

### 5. Excluded
I exclude AI traffic columns because they are mostly missing and not reliable.

### Answer
One row represents one content item per client per day.

The time window used is March 2026 (month = 2026-03).

## 2. Fields: feature / label / context / excluded

*Sort every field you plan to touch into these four buckets. Excluded needs a why.*

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


### Answer

Feature fields:
- clicks
- impressions
- position

Label:
- high_clicks (derived)

Context:
- date
- query
- page

Excluded:
- June 2026 data (to avoid leakage)

## 3. Verify it with queries (grain, counts, missing values, windows)

*Every claim above gets a query cell here. A contract claim without a query next to it is a guess.*

In [6]:
!pip install duckdb

import duckdb

con = duckdb.connect()

con.execute("INSTALL httpfs;")
con.execute("LOAD httpfs;")

# 🔐 paste your NEW token here
con.execute(f"""
CREATE SECRET hf_secret (
    TYPE HUGGINGFACE,
    TOKEN '{HF_TOKEN}'
);
""")

# ✅ LOAD ONLY ONE MONTH (IMPORTANT)
df = con.execute("""
SELECT *
FROM read_parquet(
    'hf://datasets/FlyRank/internship-warehouse/fact_content_daily_performance/month=2026-03/*.parquet'
)
LIMIT 100000
""").df()

df.head()


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,report_date,client_hash_id,content_hash_id,client_has_gsc,client_has_ga4,gsc_data_available,ga4_data_available,gsc_impressions,gsc_clicks,gsc_sum_position,...,sessions_ai,ai_chatgpt,ai_perplexity,ai_gemini,ai_copilot,ai_claude,ai_meta,ai_other,scroll_events,month
0,2026-03-01,client_73cda7b4e4f265ea,content_b7e512995f79d5a6,True,False,True,<NA>,20,0,67,...,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,2026-03
1,2026-03-01,client_73cda7b4e4f265ea,content_05597932fe4da067,True,False,True,<NA>,1,0,0,...,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,2026-03
2,2026-03-01,client_73cda7b4e4f265ea,content_7a105f548d9c6916,True,False,True,<NA>,125,1,616,...,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,2026-03
3,2026-03-01,client_73cda7b4e4f265ea,content_905aa32a0230694e,True,False,True,<NA>,7,0,28,...,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,2026-03
4,2026-03-01,client_73cda7b4e4f265ea,content_a3ea9792f793ec72,True,False,True,<NA>,11,0,25,...,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,2026-03


In [12]:
features = con.execute("""
SELECT
    gsc_impressions,
    gsc_clicks,
    gsc_sum_position,
    (gsc_clicks * 1.0 / NULLIF(gsc_impressions,0)) AS ctr,
    scroll_events
FROM df
WHERE gsc_data_available IS TRUE
LIMIT 10000
""").df()

features.head()

,gsc_impressions,gsc_clicks,gsc_sum_position,ctr,scroll_events
0,20,0,67,0.000,<NA>
1,1,0,0,0.000,<NA>
2,125,1,616,0.008,<NA>
3,7,0,28,0.000,<NA>
4,11,0,25,0.000,<NA>


In [7]:
con.execute("""
SELECT COUNT(*) AS total_rows,
       COUNT(DISTINCT client_hash_id || content_hash_id || report_date) AS unique_rows
FROM df
""").df()

,total_rows,unique_rows
0,100000,100000


In [8]:
con.execute("""
SELECT COUNT(*) AS total_rows,
       MIN(report_date) AS start_date,
       MAX(report_date) AS end_date
FROM df
""").df()

,total_rows,start_date,end_date
0,100000,2026-03-01,2026-03-03


In [9]:
con.execute("""
SELECT COUNT(*) AS total_rows,
       SUM(CASE WHEN gsc_data_available IS TRUE THEN 1 ELSE 0 END) AS available_rows
FROM df
""").df()

,total_rows,available_rows
0,100000,39751.0


### Features

1. gsc_impressions — knowable at decision time because it comes from past data.
2. gsc_clicks — known after content performance is observed.
3. gsc_sum_position — reflects search ranking at the time.
4. ctr — derived from impressions and clicks, both known.
5. scroll_events — reflects user engagement behavior.

In [10]:
df["leak_feature"] = df["gsc_clicks"]

In [11]:
df = df.drop(columns=["leak_feature"])

### Leakage Experiment
When I added the leak_feature (same as label), any model would achieve near-perfect accuracy because the answer is already included in input. This demonstrates data leakage.

### Data Limitation

The dataset has missing GA4 and AI traffic data for many rows, so analysis is limited mainly to GSC metrics.

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.